---

# 🎓 Self-Assignment: Build Your Own RAG System

## Mini Project — "Ask My Documents"

**Objective:** Build a complete, end-to-end RAG application using everything you learned in this course. You will create a system that can answer questions about **your own documents** (PDFs, text files, or web pages).

**Estimated Time:** 2–3 hours

**Difficulty:** ⭐⭐⭐ Intermediate

---

### 📋 Project Brief

> *You are an AI engineer at a company. Your team has a collection of internal documents (policies, technical guides, meeting notes). Build a RAG-powered Q&A system that lets employees ask natural language questions and get accurate, grounded answers from these documents.*

---

### ✅ Requirements

Your submission must include a working Jupyter notebook with the following **7 tasks**:

| Task | Description | Points |
|------|-------------|--------|
| **Task 1** | Load at least **3 documents** from at least **2 different sources** (e.g., PDF + Web, or PDF + TXT) | 10 |
| **Task 2** | Implement a chunking strategy with a **justified choice** of `chunk_size` and `chunk_overlap` — write a comment explaining your reasoning | 10 |
| **Task 3** | Store embeddings in a **Chroma** vector store with persistence enabled | 10 |
| **Task 4** | Build a **retriever** and demonstrate it works by showing top-K results for 3 different queries | 10 |
| **Task 5** | Write a **custom RAG prompt** that includes a system message with specific instructions (e.g., "answer in bullet points", "cite the source page", or "say I don't know if unsure") | 15 |
| **Task 6** | Assemble the **complete RAG chain** using LCEL (pipe operators) and test it with at least **5 questions** | 15 |
| **Task 7** | **Bonus Challenges** (pick at least one) — see below | 30 |

**Total: 100 points**

---

### 🌟 Task 7 — Bonus Challenges (pick at least one for full marks)

| Challenge | Description | Points |
|-----------|-------------|--------|
| **A. Multi-turn memory** | Add conversation history so follow-up questions work (e.g., "What about its pricing?" after asking about a product) | 10 |
| **B. Source citation** | Modify the chain to return **which documents** were used to answer each question (page number, filename) | 10 |
| **C. Evaluation harness** | Create 5 question-answer pairs as ground truth, then run your RAG chain and compare its answers against the ground truth (simple string match or LLM-as-judge) | 10 |
| **D. Chunking experiment** | Try 3 different chunk sizes (e.g., 200, 500, 1000) and compare retrieval quality — which size finds the best context for the same query? | 10 |
| **E. Hybrid retriever** | Combine similarity search with MMR or metadata filtering — explain when each strategy is better | 10 |
| **F. Streaming UI** | Build a simple interactive cell where users type questions and see the RAG chain stream its answer token by token | 10 |

---

### 🏗️ Starter Scaffold

The cells below provide a **skeleton structure** for your project. Each cell has `TODO` comments where you need to fill in your code. The structure follows the same 4-phase approach from the course.

> **Tip:** Refer back to the demo cells above (Cells 1–22) whenever you get stuck. The patterns are the same — you're just applying them to your own data now.

---

#### Task 1 — Load Your Documents

In [1]:
!pip install langchain_community langchain_openai langchain_chroma python-dotenv pydf pypdf langchain-text-splitters

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.5/343.5 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━

In [3]:
from langchain_community.document_loaders import WebBaseLoader
web_loader = WebBaseLoader([
    "https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel",
    "https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel_II",
    "https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel_III"
])
web_docs = web_loader.load()

from langchain_community.document_loaders import TextLoader
txt_loader = TextLoader("The_Legend_of_Heroes_Trails_of_Cold_Steel_IV_Wikipedia.txt")
txt_docs = txt_loader.load()

all_docs = web_docs + txt_docs

print(f"Web documents loaded: {len(web_docs)}")
print(f"Text documents loaded: {len(txt_docs)}")
print(f"Total: {len(all_docs)}")

Web documents loaded: 3
Text documents loaded: 1
Total: 4


#### Task 2 — Chunk Your Documents

Choose your `chunk_size` and `chunk_overlap` and **explain why** in a comment.

In [4]:
# chunk size 500 dan chunk overlap 50 cukup untuk memberi keseimbangan antara konteks
# dengan efisiensi biaya
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(all_docs)

print(f"Panjang dokumen: {len(all_docs)}")
print(f"Total chunks: {len(chunks)}")
print(f"average chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)}")

Panjang dokumen: 4
Total chunks: 185
average chunk length: 352


#### Task 3 — Store in ChromaDB

In [5]:
from langchain_chroma import Chroma
from langchain_openai import AzureOpenAIEmbeddings
from google.colab import userdata
import shutil, os

embeddings = AzureOpenAIEmbeddings(
    azure_deployment="text-embedding-3-small",
    azure_endpoint=userdata.get("AZURE_ENDPOINT"),
    api_key=userdata.get("AZURE_API"),
    chunk_size=100
)

In [6]:
PERSISTS_DIR = "./content/cold_steel_db"
if os.path.exists(PERSISTS_DIR):
    shutil.rmtree(PERSISTS_DIR)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSISTS_DIR,
    collection_name='cold_steel_all'
)

print(f"Stored {len(chunks)} chunks in Chroma")

Stored 185 chunks in Chroma


#### Task 4 — Build & Test the Retriever

In [7]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

test_queries = [
    "Who is Rean Schwarzer?",
    "What is the setting of the Erebonian Empire?",
    "How does the combat system work?",
]

for query in test_queries:
    docs = retriever.invoke(query)
    print(f"\n🔎 Query: \"{query}\"")
    for i, doc in enumerate(docs):
        print(f"   [{i+1}] {doc.page_content[:100]}...")


🔎 Query: "Who is Rean Schwarzer?"
   [1] Beginning one month after the original Trails of Cold Steel and concurrently alongside the final cha...
   [2] During a party in the Imperial Palace, Rean learns of his origins; while serving as a brigadier gene...
   [3] While teaching his students, Rean is ordered by the Imperial government to resolve numerous conflict...
   [4] The story begins two years after the events of Trails of Cold Steel II, and several months after the...

🔎 Query: "What is the setting of the Erebonian Empire?"
   [1] Plot[edit]
The game is set in the Erebonian Empire and takes place after the Trails in the Sky trilo...
   [2] During the course of the story, Class VII, while resolving numerous incidents across western Ereboni...
   [3] attack the Erebonian capital of Heimdallr to stop the Noble Alliance and end the war. They succeed, ...
   [4] Two weeks after the events of Trails of Cold Steel III, Chancellor Giliath Osborne has taken control...

🔎 Query: "How does 

#### Task 5 — Design Your Custom RAG Prompt

Create a **custom system message** that shapes how the model answers. Be creative and specific.

In [25]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are an expert on 'The Legend of Heroes: Trails of Cold Steel' series.
    Answer the question using the provided context.
    - Cite your sources clearly using the URL from the metadata.
    - If you don't know the answer, state that the information is not in the records.
    - Use bullet points for characters or gameplay mechanics.
    """),
    MessagesPlaceholder("chat_history"),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

rendered = rag_prompt.invoke({
    "context": (
        "Rean Schwarzer is the main protagonist of Trails of Cold Steel. "
        "He is a student at Thors Military Academy and a member of Class VII. "
        "Rean is known for his strong sense of justice, his role as the Ashen Chevalier, "
        "and his ability to access Spirit Unification powers in combat."
    ),
    "question": "Who is Rean Schwarzer?",
    "chat_history": []
})
for msg in rendered.messages:
    print(f"[{msg.type.upper()}\n{msg.content}]")

[SYSTEM

    You are an expert on 'The Legend of Heroes: Trails of Cold Steel' series.
    Answer the question using the provided context.
    - Cite your sources clearly using the URL from the metadata.
    - If you don't know the answer, state that the information is not in the records.
    - Use bullet points for characters or gameplay mechanics.
    ]
[HUMAN
Context:
Rean Schwarzer is the main protagonist of Trails of Cold Steel. He is a student at Thors Military Academy and a member of Class VII. Rean is known for his strong sense of justice, his role as the Ashen Chevalier, and his ability to access Spirit Unification powers in combat.

Question: Who is Rean Schwarzer?]


#### Task 6 — Assemble & Test the Full RAG Chain

In [27]:
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage

model = AzureChatOpenAI(
    azure_deployment="o4-mini",
    azure_endpoint=userdata.get("AZURE_ENDPOINT"),
    api_key=userdata.get("AZURE_API"),
    api_version="2024-12-01-preview"
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    RunnableParallel({
        "context": (lambda x: x["question"]) | retriever | format_docs,
        "question": lambda x: x["question"],
        "chat_history": lambda x: x.get("chat_history", []),
    })
    | rag_prompt
    | model
    | StrOutputParser()
)

questions = [
    "What is Class VII?",
    "Who are the main protagonists?",
    "Explain the setting of the game.",
    "What are Tactical Link systems?",
    "Which empire does the story take place in?"
]

for q in questions:
    answer = rag_chain.invoke({"question": q, "chat_history": []})
    print(f"\n{q}")
    print(f"{answer}")
    print("-" * 50)


What is Class VII?
Class VII is the central unit of students around which Trails of Cold Steel’s story revolves:

• A newly formed class at Thors Military Academy in Trista.  
• The only class at Thors not segregated by social rank—its members include both Erebonian nobles and commoners.  
• Composed of key protagonists such as Rean Schwarzer, Alisa Reinford, Elliot Craig, Laura S. Arseid, and others whose field studies expose them to the Empire’s brewing civil conflict.  

Source: https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel
--------------------------------------------------

Who are the main protagonists?
The main protagonists of The Legend of Heroes: Trails of Cold Steel are the members of Class VII (plus their instructor) at Thors Military Academy:  
• Rean Schwarzer (the player-character protagonist)  
• Alisa Reinford  
• Elliot Craig  
• Laura S. Arseid  
• Machias Regnitz  
• Jusis Albarea  
• Emma Millstein  
• Fie Claussell  
• Gaius Worzel  
• Cr

#### Task 7 — Bonus Challenge(s)

Pick **at least one** bonus challenge from the table above and implement it below.

In [33]:
print("--- Challenge A: Multi-turn Memory ---")
chat_history = []
q1 = "Who is the protagonist of Trails of Cold Steel?"
a1 = rag_chain.invoke({"question": q1, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=q1), AIMessage(content=a1)])
print(f"Q: {q1}\nA: {a1}...")

q2 = "What is his special weapon?"
a2 = rag_chain.invoke({"question": q2, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=q2), AIMessage(content=a2)])
print(f"\nFollow-up Q: {q2}\nA: {a2}...")

q3 = "Which shool does he attend?"
a3 = rag_chain.invoke({"question": q3, "chat_history": chat_history})
print(f"\nFollow-up Q: {q3}\nA: {a3}...")

--- Challenge A: Multi-turn Memory ---
Q: Who is the protagonist of Trails of Cold Steel?
A: The protagonist of Trails of Cold Steel is:

• Rean Schwarzer  
  Source: https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel...

Follow-up Q: What is his special weapon?
A: Rean’s hallmark “special weapon” is his bond with a Divine Knight:

• Valimar – the Divine Knight of the Azure Wolf that only an Awakener like Rean can summon and pilot  
  Source: https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel...

Follow-up Q: Which shool does he attend?
A: Rean Schwarzer is a student at Thors Military Academy in the city of Trista.  
Source: https://en.wikipedia.org/wiki/The_Legend_of_Heroes:_Trails_of_Cold_Steel...


In [30]:
print("--- Challenge B: Source Citation (with document metadata)")

def format_docs_with_citation(docs):
    parts = []
    for d in docs:
        src = d.metadata.get("source", "unknown")
        page = d.metadata.get("page", "N/A")
        parts.append(f"[Source: {src}] | Page: {page}]\n{d.page_content}")
    return "\n\n".join(parts)

rag_chain_with_sources = RunnableParallel(
    answer = (
        RunnableParallel({
            "context": (lambda x: x["question"]) | retriever | format_docs_with_citation,
            "question": lambda x: x["question"],
            "chat_history": lambda x: x.get("chat_history", [])
        })
        | rag_prompt
        | model
        | StrOutputParser()
    ),
    source_docs = (lambda x: x["question"]) | retriever
)

cite_query = "What is Class VII and who are its members?"
result = rag_chain_with_sources.invoke({"question": cite_query, "chat_history": []})

print(f"\n{cite_query}")
print(f"\nAnswer:\n{result['answer']}")
print("\nSources used:")

seen = set()
for doc in result["source_docs"]:
    src = doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page", "N/A")
    key = (src, page)
    if key not in seen:
        print(f" - {src} (page {page})")
        seen.add(key)

--- Challenge B: Source Citation (with document metadata)

What is Class VII and who are its members?

Answer:
Class VII is the newly formed, cross-class course at Thors Military Academy in Trista—unique in that it deliberately mixes both nobility and commoners so students can learn from one another.  It serves as the main focal “team” in Trails of Cold Steel, sending its members on field studies across Erebonia amid rising social and political tensions.[1]

Its original members are:  
• Rean Schwarzer (a commoner later adopted by a noble family)  
• Alisa Reinford (daughter of an influential engineering magnate)  
• Elliot Craig (son of a famed Imperial Army commander)  
• Laura S. Arseid (heiress to a legendary Arseid sword-fighting clan)  
• Crow Armbrust (a second-year student who joins Class VII after failing to graduate)  
• Millium Orion (an operative from the Imperial Army’s Intelligence Division)[1]

Source:  
[1] “The Legend of Heroes: Trails of Cold Steel,” Wikipedia, https:

In [31]:
print("--- Challenge C: Evaluation Harness (LLM-as-Judge)")

ground_truth = [
    {"question": "What is Rean Schwarzer's ogre power?",
     "expected": "Rean possesses the ability to tap into an Ogre/Awakener power that dramatically boosts his combat strength."},
    {"question": "Which military academy do the characters attend?",
     "expected": "They attend Thors Military Academy."},
    {"question": "What is the name of Rean's signature sword style?",
     "expected": "Rean practices the Eight Leaves One Blade sword style."},
    {"question": "Who is the antagonist in Trails of Cold Steel III?",
     "expected": "Osborne and the Ouroboros society are the primary antagonists, with Ash Carbide as a key figure."},
    {"question": "What country is the Trails of Cold Steel series set in?",
     "expected": "The games are set in the Erebonian Empire."},
]

judge_llm = model

eval_results = []
for item in ground_truth:
    rag_answer = rag_chain.invoke({"question": item["question"], "chat_history": []}) # Ensures consistent input format

    judge_prompt = (
        f"Expected key fact: {item['expected']}\n"
        f"RAG answer: {rag_answer}\n\n"
        "Does the RAG answer cover the expected key fact? "
        "Reply with exactly one word: Yes or No."
    )

    verdict_msg = judge_llm.invoke(judge_prompt)
    verdict = verdict_msg.content.strip().split()[0]
    eval_results.append({"question": item['question'], "verdict": verdict})
    print(f"{'Good' if verdict.lower() == 'yes' else 'Bad'} [{verdict}] {item['question']}")

passed = sum(1 for r in eval_results if r["verdict"].lower() == "yes")
print(f'\nScore: {passed}/{len(ground_truth)} questions passed')


--- Challenge C: Evaluation Harness (LLM-as-Judge)
Bad [No] What is Rean Schwarzer's ogre power?
Good [Yes] Which military academy do the characters attend?
Bad [No] What is the name of Rean's signature sword style?
Bad [No] Who is the antagonist in Trails of Cold Steel III?
Good [Yes] What country is the Trails of Cold Steel series set in?

Score: 2/5 questions passed


---

### 📦 Submission Checklist

Before submitting, verify the following:

- [x] **Task 1:** Loaded 3+ documents from 2+ different source types
- [x] **Task 2:** Chunking strategy implemented with a written justification comment
- [x] **Task 3:** Chroma vector store created with persistence to disk
- [x] **Task 4:** Retriever tested with 3 queries, showing top-K results with metadata
- [x] **Task 5:** Custom RAG prompt with a specific, thoughtful system message
- [x] **Task 6:** Full RAG chain assembled with LCEL pipes, tested with 5+ questions
- [x] **Task 7:** At least one bonus challenge completed
- [x] **All cells run** top-to-bottom without errors (Kernel → Restart & Run All)
- [x] **No hardcoded API keys** — uses environment variables or `.env` file

### 📁 What to Submit

1. This notebook (`.ipynb`) with all cells executed and outputs visible
2. Your `.env.example` file (with placeholder values, NOT real keys)
3. A short `README.md` (3–5 sentences) describing:
   - What documents you chose and why
   - What bonus challenge(s) you completed
   - One thing you learned or found surprising

---

### 💡 Tips for Success

- **Start with small, simple documents** — don't load 500 pages on your first try.
- **Test each task independently** before chaining them together.
- **Print intermediate results** — check what your retriever returns before building the full chain.
- **Experiment with chunk sizes** — this is the single biggest lever for RAG quality.
- **Read the error messages** — LangChain errors are usually descriptive and tell you exactly what's wrong.

Good luck! 🚀

---

## 🧹 Cleanup

In [ ]:
# ============================================================
# CLEANUP: Remove persisted Chroma database
# ============================================================
import shutil, os

if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
    print("🗑️  Removed './chroma_db/' directory")
else:
    print("ℹ️  Nothing to clean up")

print("\n✅ Demo complete! You've built a full RAG pipeline with Azure OpenAI + ChromaDB + LangChain.")